In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler
from sklearn.experimental import enable_iterative_imputer  # Nécessaire pour activer l'IterativeImputer
from sklearn.impute import IterativeImputer
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE  # Pour gérer le déséquilibre des classes

In [54]:
def data_remplace_Mode_isna(df):
    # Remplacement par la valeur la plus fréquente (Mode)
    for col in ["State", "BankState", "NewExist", "LowDoc","RevLineCr"]:
        df[col] = df[col].fillna(df[col].mode()[0])

In [55]:
def data_remplace_MIS_STATUS(df):
    for col in ["FranchiseCode", "RevLineCr", "LowDoc"]:
        df["MIS_Status"] = df.groupby(col)["MIS_Status"].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "PIF"))


In [61]:
# 📌 Charger les données
df = pd.read_csv("preparation_data.csv")
data_remplace_MIS_STATUS(df)
data_remplace_Mode_isna(df)
# ➤ Création de nouvelles variables
recession_years = [2008, 2009, 2020]
df['RecessionPeriod'] = df['ApprovalFY'].apply(lambda x: 1 if x in recession_years else 0)
df['Loan_Usage_Ratio'] = df['DisbursementGross'] / df['GrAppv']
df['Employee_Loan_Ratio'] = df['NoEmp'] / df['GrAppv']
df['JobImpact'] = df['CreateJob'] + df['RetainedJob']
df['Loan_Term_Category'] = pd.cut(df['Term'], bins=[0, 60, 120, np.inf], labels=['Court', 'Moyen', 'Long'])

# ➤ Sélection des variables
num_vars = ['ApprovalFY', 'Term', 'DisbursementGross', 'UrbanRural', 'GrAppv', 'NoEmp', 'Loan_Usage_Ratio', 'Employee_Loan_Ratio', 'JobImpact','MIS_Status']
cat_vars = ['NewExist', 'FranchiseCode', 'RevLineCr', 'LowDoc', 'RecessionPeriod', 'State', 'BankState', 'NAICS_2', 'Loan_Term_Category']

df = pd.get_dummies(df, columns=['Loan_Term_Category'], drop_first=False, dtype=int)

# ➤ Définition de X et y
X = df.drop(columns=["MIS_Status"])
y = df["MIS_Status"]

print(y.isna().sum())

# ➤ Transformation des labels de sortie en 0/1
#y = y.map({"PIF": 1, "CHGOFF": 0})

# 📌 Pipeline de prétraitement
# ⚡ Encodage des variables catégorielles
cat_pipeline = Pipeline([
    #("impute", IterativeImputer()),  # Remplissage des NaN par la valeur la plus fréquente
    ("onehot", OneHotEncoder(handle_unknown="ignore", dtype=int))  # Encodage OneHot
])

# ⚡ Normalisation des variables numériques
num_pipeline = Pipeline([
    #("impute", IterativeImputer()),  # Remplissage des NaN par la moyenne
    ("scaler", MinMaxScaler())  # Normalisation Min-Max
])

# ⚡ Clustering des États et banques avec KMeans
class KMeansTransformer:
    def __init__(self, n_clusters=10):
        self.n_clusters = n_clusters
        self.kmeans = None

    def fit(self, X, y=None):
        self.kmeans = KMeans(n_clusters=self.n_clusters, random_state=42, n_init=10)
        self.kmeans.fit(X)
        return self

    def transform(self, X):
        return self.kmeans.predict(X).reshape(-1, 1)

state_bank_pipeline = Pipeline([
    ("label_encode", LabelEncoder()),  # Conversion en numérique
    ("kmeans", KMeansTransformer(n_clusters=10))  # Clustering avec KMeans
])

# ➤ Création du ColumnTransformer pour appliquer ces transformations
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_vars),
    ("cat", cat_pipeline, cat_vars),
], remainder="drop")

# 📌 Pipeline final avec Random Forest
pipeline = Pipeline([
    ("preprocessing", preprocessor), # Oversampling de la classe minoritaire
    ("classifier", RandomForestClassifier(class_weight="balanced", n_estimators=200, max_depth=15, random_state=42))
])



6007


In [57]:
# 📌 Split des données
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)#, stratify=y)

# 📌 Entraînement du modèle

In [58]:
pipeline.fit(X_train, y_train)

# 📌 Prédiction et évaluation
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# 📌 Importance des features
model = pipeline.named_steps["classifier"]
importances = model.feature_importances_
features = X_train.columns

# 📌 Affichage de l'importance des features
plt.figure(figsize=(10, 5))
sns.barplot(x=importances, y=features)
plt.title("Importance des Features")
plt.show()


ValueError: A given column is not a column of the dataframe

In [ ]:
df.isna().sum()

Term                           0
FranchiseCode                  0
State                          0
BankState                      0
NAICS_2                        0
NoEmp                          0
NewExist                       0
RetainedJob                    0
CreateJob                      0
UrbanRural                     0
RevLineCr                      0
ApprovalFY                     0
DisbursementGross              0
GrAppv                         0
LowDoc                         0
MIS_Status                  6007
RecessionPeriod                0
Loan_Usage_Ratio               0
Employee_Loan_Ratio            0
JobImpact                      0
Loan_Term_Category_Court       0
Loan_Term_Category_Moyen       0
Loan_Term_Category_Long        0
dtype: int64